<a href="https://colab.research.google.com/github/tsuji-mutsushi/yolov3-finetuning/blob/main/yolov3_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

/content
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to trash_catch-4 in darknet:: 100%|██████████| 10258/10258 [00:01<00:00, 6622.48it/s]


データセットの場所: /content/trash_catch-4


# darknetのクローン

In [77]:
%cd /content/drive/MyDrive
!rm -rf darknet

/content/drive/MyDrive


In [2]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [80]:
# Google Drive上の作業ディレクトリに移動
%cd '/content/drive/My Drive'

# darknetをクローン
!git clone https://github.com/AlexeyAB/darknet.git
%cd darknet
#作業ファイルを作成
!mkdir trash_catch

/content/drive/My Drive
fatal: destination path 'darknet' already exists and is not an empty directory.
/content/drive/My Drive/darknet
mkdir: cannot create directory ‘trash_catch’: File exists


# データの準備

In [81]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 102.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [82]:
API_KEY = "GyXkbhoTohOTow40yhKc"
WORKSPACE    = "mutsushi-tsuji-zkndc"
PROJECT      = "trash_catch-3qcwd"
FORMAT       = "darknet"
VERSION      = 4

In [ ]:
from roboflow import Roboflow
%cd /content/drive/MyDrive/darknet/trash_catch

# データセットのダウンロード
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download(FORMAT)

print(f"データセットの場所: {dataset.location}")

/content/drive/MyDrive/darknet/trash_catch
loading Roboflow workspace...
loading Roboflow project...


# yolov3の学習準備

In [4]:
# 作業用ディレクトリ作成
!mkdir -p trash_catch/data
!mkdir -p trash_catch/backup

# 現在のディレクトリ確認
!pwd

/content/drive/My Drive/darknet


In [8]:
from os import terminal_size
# クラス名を記載（1行に1クラス）
!cd trash_catch/ && echo -e 'can\npaper\nplastic_bottle' > obj.names

In [5]:
!cat trash_catch/obj.names

can
paper
plastic_bottle


In [12]:
!pwd

/content/drive/MyDrive/darknet


In [32]:
!cp cfg/yolov3.cfg trash_catch/yolov3-obj.cfg

In [33]:
# クラス数を設定
num_classes = 3

# 計算
max_batches = max(6000, num_classes * 2000)
steps1 = int(max_batches * 0.8)
steps2 = int(max_batches * 0.9)
filters = (num_classes + 5) * 3

print(f"設定値: classes={num_classes}, filters={filters}")
print()

# cfgファイルを読み込み
with open('trash_catch/yolov3-obj.cfg', 'r') as f:
    lines = f.readlines()

# === 変更する行を直接指定 ===
# （YOLOv3標準cfgの場合）

# batch, subdivisions, max_batches, steps
lines[2] = 'batch=64\n'
lines[3] = 'subdivisions=16\n'
lines[19] = f'max_batches={max_batches}\n'
lines[21] = f'steps={steps1},{steps2}\n'

# YOLO層1の設定（1つ目のYOLO層）
lines[602] = f'filters={filters}\n'  # filters（YOLO層の直前）
lines[609] = f'classes={num_classes}\n'  # classes（YOLO層内）

# YOLO層2の設定（2つ目のYOLO層）
lines[688] = f'filters={filters}\n'
lines[695] = f'classes={num_classes}\n'

# YOLO層3の設定（3つ目のYOLO層）
lines[775] = f'filters={filters}\n'
lines[782] = f'classes={num_classes}\n'

# 保存
with open('trash_catch/yolov3-obj.cfg', 'w') as f:
    f.writelines(lines)
print(f"\n✓ yolov3-obj.cfg の編集が完了しました！")

設定値: classes=3, filters=24


✓ yolov3-obj.cfg の編集が完了しました！


In [34]:
!cat trash_catch/yolov3-obj.cfg

[net]
# Testing
batch=64
subdivisions=16
# Training
# batch=64
# subdivisions=16
width=416
height=416
channels=3
momentum=0.9
decay=0.0005
angle=0
saturation = 1.5
exposure = 1.5
hue=.1

learning_rate=0.001
burn_in=1000
max_batches=6000
policy=steps
steps=4800,5400
scales=.1,.1

[convolutional]
batch_normalize=1
filters=32
size=3
stride=1
pad=1
activation=leaky

# Downsample

[convolutional]
batch_normalize=1
filters=64
size=3
stride=2
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=64
size=3
stride=1
pad=1
activation=leaky

[shortcut]
from=-3
activation=linear

# Downsample

[convolutional]
batch_normalize=1
filters=128
size=3
stride=2
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=64
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=128
size=3
stride=1
pad=1
activation=leaky

[shortcut]
from=-3
activation=linear

[convolutional]
bat

In [ ]:
#ルートディレクトリに移動
%cd \/content
# データファイルをコピー
!cp -r /content/trash_catch-4/ /content/drive/MyDrive/darknet/trash_catch

In [14]:
#darknetディレクトリに戻る
%cd \/content/drive/MyDrive/darknet

/content/drive/MyDrive/darknet


In [51]:
import glob
import os

# train.txt作成（trainフォルダ内の画像）
train_images = glob.glob('trash_catch/trash_catch-4/train/*.jpg')
with open('trash_catch/train.txt', 'w') as f:
    for img_path in train_images:
        abs_path = os.path.abspath(img_path)
        f.write(abs_path + '\n')

# test.txt作成
test_images = glob.glob('trash_catch/trash_catch-4/test/*.jpg')
with open('trash_catch/test.txt', 'w') as f:
    for img_path2 in test_images:
        abs_path = os.path.abspath(img_path2)
        f.write(abs_path + '\n')

print(f"✓ 訓練データ: {len(train_images)}枚")
print(f"✓ 検証データ: {len(test_images)}枚")

# 中身を確認
print("\n--- train.txt の先頭5行 ---")
!head -5 trash_catch/train.txt
print("\n--- test.txt の先頭5行 ---")
!head -5 trash_catch/test.txt

✓ 訓練データ: 4546枚
✓ 検証データ: 240枚

--- train.txt の先頭5行 ---
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/train/paper9_jpg.rf.8d2b8607e90cfe1e159f0c0c15634ca0.jpg
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/train/paper9_jpg.rf.fca90c79c6a5b88d286a9aacb7883036.jpg
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/train/pexels-anna-shvets-5029862_jpg.rf.3911bdddeeb4410a354b6c602732c2b2.jpg
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/train/pexels-anna-shvets-5029862_jpg.rf.4a76e3f85807f6db7935bf2dd360abc9.jpg
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/train/pexels-karolina-grabowska-4498089_jpg.rf.34c7f30a9a251c5bb8abf50f8c921f1f.jpg

--- test.txt の先頭5行 ---
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/test/-18_jpg.rf.80c8e25c5f99885f875a4e20d651f31a.jpg
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/test/-216_jpg.rf.aea0c5760207b93a980377d5990d7ab3.jpg
/content/drive/MyDrive/darknet/trash_catch/trash_catch-4/test/-2

In [16]:
%%writefile trash_catch/obj.data
classes=3
train=trash_catch/train.txt
valid=trash_catch/test.txt
names=trash_catch/obj.names
backup=trash_catch/backup/

Writing trash_catch/obj.data


In [52]:
# darknet53の事前学習済み重みをダウンロード
!wget https://pjreddie.com/media/files/darknet53.conv.74

--2025-10-27 09:36:57--  https://pjreddie.com/media/files/darknet53.conv.74
Resolving pjreddie.com (pjreddie.com)... 104.21.88.156, 172.67.185.199, 2606:4700:3030::ac43:b9c7, ...
Connecting to pjreddie.com (pjreddie.com)|104.21.88.156|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘darknet53.conv.74.3’

darknet53.conv.74.3     [ <=>                ]   8.84K  --.-KB/s    in 0.004s  

2025-10-27 09:36:58 (2.35 MB/s) - ‘darknet53.conv.74.3’ saved [9053]



In [53]:
# Makefileを編集（GPU有効化）
makefile_content = """
GPU=1
CUDNN=1
CUDNN_HALF=1
OPENCV=1
"""

# Makefileの先頭を書き換え
with open('Makefile', 'r') as f:
    lines = f.readlines()

lines[0] = 'GPU=1\n'
lines[1] = 'CUDNN=1\n'
lines[2] = 'CUDNN_HALF=1\n'
lines[3] = 'OPENCV=1\n'

with open('Makefile', 'w') as f:
    f.writelines(lines)

print("Makefile を編集しました")

Makefile を編集しました


In [58]:
!chmod +x darknet

In [59]:
!make -j4

chmod +x *.sh


In [60]:
!export LD_LIBRARY_PATH="/usr/local/cuda/compat:${LD_LIBRARY_PATH}"

In [61]:
# 学習実行（-map オプションで精度グラフも保存）
!./darknet detector train trash_catch/obj.data trash_catch/yolov3-obj.cfg darknet53.conv.74 -map -dont_show

 CUDA-version: 12050 (12040)
, cuDNN: 9.2.1, CUDNN_HALF=1, GPU count: 1  
 CUDNN_HALF=1 
 OpenCV version: 4.5.4
 Prepare additional network for mAP calculation...
 0 : compute_capability = 750, cudnn_half = 1, GPU: Tesla T4 
net.optimized_memory = 0 
mini_batch = 1, batch = 16, time_steps = 1, train = 0 
   layer   filters  size/strd(dil)      input                output
   0 Create CUDA-stream - 0 
 Create cudnn-handle 0 
conv     32       3 x 3/ 1    416 x 416 x   3 ->  416 x 416 x  32 0.299 BF
   1 conv     64       3 x 3/ 2    416 x 416 x  32 ->  208 x 208 x  64 1.595 BF
   2 conv     32       1 x 1/ 1    208 x 208 x  64 ->  208 x 208 x  32 0.177 BF
   3 conv     64       3 x 3/ 1    208 x 208 x  32 ->  208 x 208 x  64 1.595 BF
   4 Shortcut Layer: 1,  wt = 0, wn = 0, outputs: 208 x 208 x  64 0.003 BF
   5 conv    128       3 x 3/ 2    208 x 208 x  64 ->  104 x 104 x 128 1.595 BF
   6 conv     64       1 x 1/ 1    104 x 104 x 128 ->  104 x 104 x  64 0.177 BF
   7 conv    128       

In [ ]:
# ========== 完全自動学習再開スクリプト ==========

import glob
import os
from google.colab import drive

# Google Driveマウント
drive.mount('/content/drive')

# darknetディレクトリに移動
%cd '/content/drive/My Drive/darknet'

# backupディレクトリの確認
backup_dir = 'trash_catch/backup'
if not os.path.exists(backup_dir):
    print(f"❌ {backup_dir} が見つかりません")
    exit()

# weightファイルを検索
weight_files = glob.glob(f'{backup_dir}/*.weights')

if not weight_files:
    print("❌ weightファイルが見つかりません。新規学習を開始してください。")
    print("\n新規学習コマンド:")
    print("!./darknet detector train trash_catch/obj.data trash_catch/yolov3-obj.cfg darknet53.conv.74 -dont_show -map")
else:
    # 最新のweightファイルを取得
    latest_weight = max(weight_files, key=os.path.getmtime)

    # 情報表示
    print("=" * 60)
    print("学習再開の準備")
    print("=" * 60)
    print(f"\n✓ 最新のweight: {latest_weight}")

    size_mb = os.path.getsize(latest_weight) / (1024 * 1024)
    print(f"  サイズ: {size_mb:.2f} MB")

    import datetime
    mtime = datetime.datetime.fromtimestamp(os.path.getmtime(latest_weight))
    print(f"  更新日時: {mtime}")

    # その他のweightファイルも表示
    print(f"\n【backup内の全weightファイル】")
    !ls -lht {backup_dir}/*.weights

    print("\n" + "=" * 60)
    print("学習を再開します...")
    print("=" * 60 + "\n")

    # 学習再開
    !./darknet detector train trash_catch/obj.data trash_catch/yolov3-obj.cfg {latest_weight} -dont_show -map

ストリーミング出力は最後の 5000 行に切り捨てられました。
 total_bbox = 893094, rewritten_bbox = 0.000000 % 
v3 (mse loss, Normalizer: (iou: 0.75, obj: 1.00, cls: 1.00) Region 82 Avg (IOU: 0.000000), count: 9, class_loss = -nan, iou_loss = nan, total_loss = nan 
v3 (mse loss, Normalizer: (iou: 0.75, obj: 1.00, cls: 1.00) Region 94 Avg (IOU: 0.000000), count: 1, class_loss = -nan, iou_loss = -nan, total_loss = -nan 
v3 (mse loss, Normalizer: (iou: 0.75, obj: 1.00, cls: 1.00) Region 106 Avg (IOU: 0.000000), count: 1, class_loss = -nan, iou_loss = -nan, total_loss = -nan 
 total_bbox = 893103, rewritten_bbox = 0.000000 % 
v3 (mse loss, Normalizer: (iou: 0.75, obj: 1.00, cls: 1.00) Region 82 Avg (IOU: 0.000000), count: 20, class_loss = -nan, iou_loss = nan, total_loss = nan 
v3 (mse loss, Normalizer: (iou: 0.75, obj: 1.00, cls: 1.00) Region 94 Avg (IOU: 0.000000), count: 10, class_loss = -nan, iou_loss = nan, total_loss = nan 
v3 (mse loss, Normalizer: (iou: 0.75, obj: 1.00, cls: 1.00) Region 106 Avg (IOU: 0.000000

In [ ]:
print("【STEP 1】単一画像テスト")
!./darknet detector test trash_catch/obj.data trash_catch/yolov3-obj.cfg trash_catch/backup/yolov3-obj_final.weights trash_catch/data/test/sample.jpg
display(Image('predictions.png'))

In [9]:
%cd /content/drive/MyDrive/darknet

/content/drive/MyDrive/darknet


In [15]:
print("\n【STEP 2】精度評価（mAP）")
!./darknet detector map trash_catch/obj.data trash_catch/yolov3-obj.cfg trash_catch/backup/yolov3-obj_last.weights


【STEP 2】精度評価（mAP）
 CUDA-version: 12050 (12040)
, cuDNN: 9.2.1, CUDNN_HALF=1, GPU count: 1  
 CUDNN_HALF=1 
 OpenCV version: 4.5.4
 0 : compute_capability = 750, cudnn_half = 1, GPU: Tesla T4 
net.optimized_memory = 0 
mini_batch = 1, batch = 16, time_steps = 1, train = 0 
   layer   filters  size/strd(dil)      input                output
   0 Create CUDA-stream - 0 
 Create cudnn-handle 0 
conv     32       3 x 3/ 1    416 x 416 x   3 ->  416 x 416 x  32 0.299 BF
   1 conv     64       3 x 3/ 2    416 x 416 x  32 ->  208 x 208 x  64 1.595 BF
   2 conv     32       1 x 1/ 1    208 x 208 x  64 ->  208 x 208 x  32 0.177 BF
   3 conv     64       3 x 3/ 1    208 x 208 x  32 ->  208 x 208 x  64 1.595 BF
   4 Shortcut Layer: 1,  wt = 0, wn = 0, outputs: 208 x 208 x  64 0.003 BF
   5 conv    128       3 x 3/ 2    208 x 208 x  64 ->  104 x 104 x 128 1.595 BF
   6 conv     64       1 x 1/ 1    104 x 104 x 128 ->  104 x 104 x  64 0.177 BF
   7 conv    128       3 x 3/ 1    104 x 104 x  64 ->  